# 🏆 Chapter 5: Best Practices & Tips

Write better notebooks, avoid common pitfalls, and level up your Jupyter game.

| # | Topic |
|---|-------|
| 1 | Notebook organization |
| 2 | Version control with notebooks |
| 3 | Common pitfalls |
| 4 | Exporting notebooks |
| 5 | Jupyter extensions |
| 6 | When NOT to use notebooks |

---
## 5.1 Notebook Organization

### The Ideal Notebook Structure

```
┌─────────────────────────────────────────────┐
│  # Title                                     │  ← Clear, descriptive title
│  ## Author, Date, Purpose                    │  ← Metadata
├─────────────────────────────────────────────┤
│  ## 1. Setup                                 │
│  import ...                                  │  ← ALL imports at the top
│  CONSTANTS = ...                             │  ← Configuration
├─────────────────────────────────────────────┤
│  ## 2. Data Loading                          │
│  data = load(...)                            │  ← Load & validate data
├─────────────────────────────────────────────┤
│  ## 3. Analysis                              │
│  results = analyze(data)                     │  ← Core work
├─────────────────────────────────────────────┤
│  ## 4. Results                               │
│  display(results)                            │  ← Visualizations & tables
├─────────────────────────────────────────────┤
│  ## 5. Conclusions                           │
│  Key findings and next steps                 │  ← Summary
└─────────────────────────────────────────────┘
```

### Best Practices

1. **Imports at the top** — all in one cell, easy to see dependencies
2. **One logical step per cell** — don't cram too much into one cell
3. **Markdown before code** — explain *what* and *why* before *how*
4. **Clear section headers** — use `## Section Name` for navigation
5. **Restart & Run All** — verify the notebook runs top-to-bottom
6. **Name notebooks well** — `01_data_cleaning.ipynb`, not `Untitled17.ipynb`

In [ ]:
# ✅ GOOD: Organized imports cell

# Standard library
import os
import sys
import json
import csv
from datetime import datetime
from collections import Counter, defaultdict

# Configuration
DATA_DIR = 'data'
OUTPUT_DIR = 'output'
RANDOM_SEED = 42

print('✅ All imports and configuration loaded successfully')
print(f'   Python version: {sys.version.split()[0]}')
print(f'   Working directory: {os.getcwd()}')

---
## 5.2 Version Control with Notebooks

### The Problem

`.ipynb` files are **JSON** — version control diffs are messy:
- Cell outputs (images, data) bloat the repo
- Execution counts change on every run
- Merge conflicts are painful

### Solutions

#### 1. `.gitignore` Patterns

```gitignore
# Jupyter checkpoint files
.ipynb_checkpoints/

# Large data files
*.csv
*.xlsx
data/

# Output files
output/
```

#### 2. `nbstripout` — Strip Output Before Committing

```bash
# Install
pip install nbstripout

# Setup as git filter (automatic)
nbstripout --install

# Or strip manually
nbstripout notebook.ipynb
```

This automatically strips all outputs and execution counts before committing.

#### 3. `jupytext` — Pair Notebooks with Scripts

```bash
pip install jupytext

# Convert notebook to .py script
jupytext --to py notebook.ipynb

# Pair: keep .ipynb and .py in sync
jupytext --set-formats ipynb,py notebook.ipynb
```

Version control the `.py` file, which has clean diffs.

#### 4. Review Tools

| Tool | Purpose |
|------|---------|
| `nbdime` | Diff & merge notebooks |
| `ReviewNB` | GitHub notebook review (web) |
| `nbstripout` | Strip outputs pre-commit |

In [ ]:
# Demo: What a notebook looks like as JSON (why diffs are messy)
import json

sample_cell = {
    "cell_type": "code",
    "source": ["x = 42\n", "print(x)"],
    "execution_count": 17,  # ← Changes every run!
    "outputs": [
        {
            "output_type": "stream",
            "name": "stdout",
            "text": ["42\n"]  # ← Output stored in JSON
        }
    ]
}

print('A code cell in JSON (this is what Git sees):')
print(json.dumps(sample_cell, indent=2))

print('\n⚠️  Every time you run a cell, execution_count and outputs change.')
print('   This creates noisy diffs in version control!')

---
## 5.3 Common Pitfalls

### ⚠️ Pitfall 1: Out-of-Order Execution

Notebooks let you run cells in **any order**. This is powerful but dangerous:

```
Cell [1]: x = 10
Cell [3]: x = x * 2      # Ran THIRD — x is now 20
Cell [2]: print(x)       # Ran SECOND — prints 10
Cell [4]: print(x)       # Ran FOURTH — prints 20
```

**Fix:** Regularly do **Kernel → Restart & Run All** to verify linear execution.

### ⚠️ Pitfall 2: Hidden State

Variables persist even after you **delete** the cell that created them:

```
Cell [1]: secret_key = 'abc123'   # You delete this cell later
Cell [2]: print(secret_key)        # Still works! Variable is in memory
```

**Fix:** Restart kernel to clear all state.

### ⚠️ Pitfall 3: Long-Running Cells

A cell that takes minutes can freeze your workflow.

**Fix:** 
- Use `%%time` to identify slow cells
- Add progress indicators
- Use `I, I` (double-I) to interrupt

### ⚠️ Pitfall 4: Massive Outputs

Printing 1 million rows will crash your browser.

**Fix:** Always limit output — use `[:10]`, `.head()`, or `print(len(data))`

### ⚠️ Pitfall 5: Not Pinning Dependencies

Your notebook works today but breaks next month when packages update.

**Fix:** Include a `requirements.txt` or pin versions in comments.

In [ ]:
# Demo: Detecting out-of-order execution
# This helper function checks if cells ran in order

def check_execution_order(notebook_path):
    """Check if a notebook was executed in linear order."""
    import json
    
    try:
        with open(notebook_path) as f:
            nb = json.load(f)
    except FileNotFoundError:
        print(f'File not found: {notebook_path}')
        return
    
    exec_counts = []
    for cell in nb['cells']:
        if cell['cell_type'] == 'code':
            ec = cell.get('execution_count')
            if ec is not None:
                exec_counts.append(ec)
    
    if not exec_counts:
        print('No executed cells found.')
        return
    
    is_ordered = all(a < b for a, b in zip(exec_counts, exec_counts[1:]))
    
    if is_ordered:
        print(f'✅ All {len(exec_counts)} cells ran in order: {exec_counts}')
    else:
        print(f'⚠️  Out-of-order execution detected!')
        print(f'   Execution counts: {exec_counts}')
        for i in range(len(exec_counts) - 1):
            if exec_counts[i] >= exec_counts[i + 1]:
                print(f'   Problem: Cell with count {exec_counts[i + 1]} '
                      f'ran after cell with count {exec_counts[i]}')

# Check this notebook (will show 'null' since we haven't run it yet)
print('This notebook has not been executed yet (all counts are null).')
print('After running it, use check_execution_order() to verify.')

---
## 5.4 Exporting Notebooks

### `nbconvert` — Convert Notebooks to Other Formats

```bash
# To HTML (most common)
jupyter nbconvert --to html notebook.ipynb

# To PDF (requires LaTeX or use --no-input for code-free)
jupyter nbconvert --to pdf notebook.ipynb

# To Python script
jupyter nbconvert --to script notebook.ipynb

# To Markdown
jupyter nbconvert --to markdown notebook.ipynb

# To slides (reveal.js)
jupyter nbconvert --to slides notebook.ipynb --post serve

# Hide code cells (show only output)
jupyter nbconvert --to html --no-input notebook.ipynb
```

### Export Formats

| Format | Command | Best For |
|--------|---------|----------|
| HTML | `--to html` | Sharing, web publishing |
| PDF | `--to pdf` | Formal reports |
| Python | `--to script` | Extracting code |
| Markdown | `--to markdown` | Documentation |
| Slides | `--to slides` | Presentations |
| LaTeX | `--to latex` | Academic papers |

### Programmatic Export

In [ ]:
# Convert a notebook to Python script programmatically
import json

def notebook_to_script(notebook_path, output_path=None):
    """Extract all code cells from a notebook into a Python script."""
    with open(notebook_path) as f:
        nb = json.load(f)
    
    lines = [
        f'# Converted from: {notebook_path}',
        f'# Auto-generated Python script',
        '',
    ]
    
    for i, cell in enumerate(nb['cells']):
        if cell['cell_type'] == 'code':
            source = ''.join(cell['source']).strip()
            if source:  # skip empty cells
                lines.append(f'# --- Cell {i} ---')
                lines.append(source)
                lines.append('')
        elif cell['cell_type'] == 'markdown':
            # Include markdown as comments
            source = ''.join(cell['source']).strip()
            first_line = source.split('\n')[0]
            if first_line.startswith('#'):
                lines.append(f'# {first_line}')
                lines.append('')
    
    script = '\n'.join(lines)
    
    if output_path:
        with open(output_path, 'w') as f:
            f.write(script)
        print(f'✅ Saved script to {output_path}')
    
    return script

# Demo: convert the first chapter
try:
    script = notebook_to_script('01_jupyter_fundamentals.ipynb')
    print('First 20 lines of converted script:')
    print('-' * 40)
    for line in script.split('\n')[:20]:
        print(line)
except FileNotFoundError:
    print('Run this from the Jupyter course directory.')

---
## 5.5 Jupyter Extensions Overview

### Classic Notebook Extensions (nbextensions)

```bash
pip install jupyter_contrib_nbextensions
jupyter contrib nbextension install --user
```

| Extension | Purpose |
|-----------|---------|
| **Table of Contents** | Auto-generate TOC from headers |
| **ExecuteTime** | Show cell execution time |
| **Variable Inspector** | GUI variable explorer |
| **Codefolding** | Collapse code blocks |
| **Autopep8** | Auto-format code to PEP 8 |
| **Scratchpad** | Quick scratch cell (Ctrl+B) |
| **Collapsible Headings** | Collapse sections |

### JupyterLab Extensions

JupyterLab uses a different extension system:

```bash
# Install via pip (most modern extensions)
pip install jupyterlab-git
pip install jupyterlab-code-formatter

# Or via jupyter (older extensions)
jupyter labextension install @jupyterlab/toc
```

| Extension | Purpose |
|-----------|---------|
| **jupyterlab-git** | Git integration in sidebar |
| **jupyterlab-lsp** | Language Server Protocol (autocomplete, linting) |
| **jupyterlab-code-formatter** | Code formatting (black, isort) |
| **jupyterlab-system-monitor** | CPU/RAM monitoring |
| **jupyterlab-drawio** | Diagram editor |

### Alternative Jupyter Interfaces

| Interface | Description |
|-----------|---------|
| **JupyterLab** | Full IDE-like experience |
| **Google Colab** | Free cloud notebooks with GPUs |
| **VS Code Notebooks** | Jupyter inside VS Code |
| **Databricks** | Enterprise notebooks for big data |
| **Kaggle Notebooks** | Cloud notebooks for competitions |

---
## 5.6 When NOT to Use Notebooks

Notebooks are fantastic tools, but they're **not always the right choice**.

### ❌ Don't Use Notebooks For:

| Scenario | Why Not | Use Instead |
|----------|---------|-------------|
| **Production code** | Hard to test, hidden state, no imports | `.py` modules + scripts |
| **Large projects** | No refactoring tools, poor modularization | IDE (PyCharm, VS Code) |
| **CI/CD pipelines** | Difficult to automate | Python scripts |
| **Team code reviews** | Messy JSON diffs | Regular `.py` files |
| **Long-running jobs** | Browser timeout, lost state | CLI scripts, Airflow |
| **Reusable libraries** | Can't import cells | Python packages |

### ✅ DO Use Notebooks For:

| Scenario | Why |
|----------|-----|
| **Exploration** | Interactive, see results immediately |
| **Prototyping** | Quick iteration on ideas |
| **Data analysis** | Narrative + code + visuals |
| **Teaching** | Students can follow along |
| **Reports** | Reproducible, shareable documents |
| **Demos** | Show code and results together |

### 💡 The Hybrid Approach

The best practice is often a **hybrid**:

1. **Explore** in a notebook
2. **Extract** working code into `.py` modules
3. **Import** modules back into the notebook
4. **Test** the modules with pytest
5. **Deploy** the modules, keep notebooks for documentation

```
project/
├── notebooks/          # Exploration & reports
│   ├── 01_explore.ipynb
│   └── 02_report.ipynb
├── src/                # Production code
│   ├── __init__.py
│   ├── data.py
│   └── models.py
├── tests/              # Tests
│   └── test_data.py
└── requirements.txt
```

In [ ]:
# Example: The Hybrid Approach in Action

# In a real project, you'd have this in src/utils.py:
# ─────────────────────────────────────────────

def compute_statistics(values):
    """Compute basic statistics for a list of numbers."""
    import statistics
    
    if not values:
        return {}
    
    return {
        'count': len(values),
        'mean': statistics.mean(values),
        'median': statistics.median(values),
        'stdev': statistics.stdev(values) if len(values) > 1 else 0,
        'min': min(values),
        'max': max(values),
    }


def format_stats_table(stats, title='Statistics'):
    """Format statistics as a readable table."""
    lines = [f'\n{title}', '=' * 35]
    for key, value in stats.items():
        if isinstance(value, float):
            lines.append(f'  {key:10s}: {value:>12,.2f}')
        else:
            lines.append(f'  {key:10s}: {value:>12,}')
    lines.append('=' * 35)
    return '\n'.join(lines)


# In the notebook, you'd import and use:
# from src.utils import compute_statistics, format_stats_table

# Demo
import random
random.seed(42)
sample_data = [random.gauss(100, 15) for _ in range(50)]

stats = compute_statistics(sample_data)
print(format_stats_table(stats, title='Sample Data Statistics'))

---
## 📝 Chapter 5 Summary

| Topic | Key Takeaway |
|---------|--------------|
| **Organization** | Imports at top, sections with headers, one step per cell |
| **Version Control** | Use `nbstripout`, `.gitignore`, or `jupytext` |
| **Pitfalls** | Out-of-order execution, hidden state, massive outputs |
| **Exporting** | `nbconvert` → HTML, PDF, Python, Slides |
| **Extensions** | Table of Contents, Variable Inspector, Git integration |
| **When NOT** | Production code, large projects, CI/CD, libraries |

---
## 🎓 Course Complete!

Congratulations! You've completed the **Jupyter Notebook Fundamentals** course.

### What You Learned:

| Chapter | Topic |
|---------|-------|
| **1** | Jupyter history, architecture, shortcuts, cell types, magics |
| **2** | Markdown, LaTeX, `IPython.display`, rich output |
| **3** | Productivity magics: timeit, store, writefile, capture, debug |
| **4** | Complete data analysis workflow with standard library |
| **5** | Best practices, version control, exporting, when (not) to use notebooks |

### Next Steps:

1. 🔬 **Practice** — Create your own analysis notebooks
2. 📚 **Explore** — Try JupyterLab, Google Colab, VS Code notebooks
3. 🧪 **Experiment** — Install extensions, customize your workflow
4. 🤝 **Share** — Export and share notebooks with colleagues
5. 🏗️ **Build** — Adopt the hybrid approach for real projects

Happy Notebooking! 📓✨